# 03 · Filter & Rank — run the shared multi-layer filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 02** you build `fp.Design` objects (`design_type="monomer"`) from each
swept sequence's recapitulation scRMSD/pLDDT + solubility proxies, run the pipeline, and report
survival (D3 part 1).

Run `00`–`02` first so `results/sequences.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — the SAME filter every project uses. We run it in
`design_type="monomer"` mode.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("DEFAULT_CUTOFFS['monomer']:", fp.DEFAULT_CUTOFFS["monomer"])

## Recapitulate the swept sequences (mock here; ESMFold/AF2 on Colab)

`results/sequences.csv` from nb 02 has the sequences but not yet their recapitulation. Fill scRMSD +
pLDDT. Here a deterministic **mock predict** stands in so the filter runs anywhere; on Colab replace
it with a real `predict(seq, tool="esmfold")` (triage) → scRMSD vs the input backbone.

In [ ]:
import hashlib
seqs = pd.read_csv("results/sequences.csv")

def mock_recapitulate(sequence):
    """Deterministic stand-in for ESMFold/AF2. NOT real. EXAMPLE_DATA only."""
    h = int(hashlib.sha256(str(sequence).encode()).hexdigest(), 16)
    return round(0.8 + (h % 350) / 100.0, 2), round(60 + (h % 40), 1)

scr, pl = zip(*[mock_recapitulate(s) for s in seqs["sequence"]])
seqs["scrmsd"], seqs["plddt"] = scr, pl
seqs.to_csv("results/sequences.csv", index=False)   # persist the recapitulation columns
print("recapitulated (mock):", seqs.shape, "| scRMSD range",
      round(seqs.scrmsd.min(),2), "-", round(seqs.scrmsd.max(),2))

## Build `fp.Design` objects (monomer)

Map each swept sequence onto a `fp.Design`: recapitulation `scrmsd`/`plddt`, the solubility proxy as
`solubility` (the filter's physics layer reads it), and stash the setting + proxies in `extra` so we
can group by setting in notebook 04. No `pae_interaction` — this is a monomer.

In [ ]:
designs = []
for _, r in seqs.iterrows():
    designs.append(fp.Design(
        design_id=f"{r['backbone']}|T{r['temperature']}|N{r['noise']}|S{r['n_seqs']}|i{r['seq_index']}",
        sequence=str(r["sequence"]),
        design_type="monomer",
        scrmsd=float(r["scrmsd"]),
        plddt=float(r["plddt"]),
        scrmsd_orthogonal=float(r["scrmsd"]),    # mock: orthogonal == primary; use ESMFold-vs-AF2 for real
        solubility=float(r["camsol_like"]),       # CamSol-STYLE heuristic, NOT real CamSol
        extra=dict(temperature=r["temperature"], noise=r["noise"], n_seqs=r["n_seqs"],
                   net_charge=r["net_charge"], hydrophobic_fraction=r["hydrophobic_fraction"]),
    ))
print(len(designs), "monomer Design objects built")

## Run the pipeline (`design_type="monomer"`)

`run_pipeline` applies the layers in order and returns a ranked DataFrame; `report` prints the
hit-rate accounting and the survival-at-each-layer figure. Monomer cutoffs: scRMSD < 2 Å,
pLDDT > 85 (solubility is a soft physics-layer signal — see `MANUAL.md §4`).

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="monomer", use_layers=(1, 2, 3))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj02")
top

## Survival by setting (honest accounting)

The headline of Project 02 is *which settings survive*. Recover the setting from each `design_id`
and report the per-setting survivor count — the seed of the Pareto analysis in notebook 04.

In [ ]:
import pandas as pd
r = df_ranked.copy()
parts = r["design_id"].str.split("|", expand=True)
r["temperature"] = parts[1].str[1:].astype(float)
r["noise"] = parts[2].str[1:].astype(float)
r["n_seqs"] = parts[3].str[1:].astype(int)
survivors = r[r["layers_passed"] >= 1]
print("survivors (passed L1 self-consistency) by temperature × noise:")
print(survivors.groupby(["temperature", "noise"]).size().unstack(fill_value=0))

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module (not a one-off script).
- [ ] Survival-at-each-layer reported; per-setting survivor counts tabulated.
- [ ] Mapping assumptions written down (scRMSD source, `solubility = camsol_like` heuristic, monomer cutoffs).

**Next:** `04_validate.ipynb` — per-setting trade-offs + the Pareto map.